In [ ]:
import pandas as pd
from pathlib import Path

RANDOM_SEED = 42
N_SAMPLE = 100

# Anpassen: nimm eine Datei, in der alle Argumente mit Originalwahrscheinlichkeit enthalten sind.
# In data folder legen, Pfad final anpassen
SOURCE_PATH = Path("true_positive_arguments.csv")

df = pd.read_csv(SOURCE_PATH)
TEXT_COL = "post_text"

print(df.columns.tolist())
df

['post_id', 'source_dataset', 'issue', 'post_text', 'Inappropriateness', 'Toxic Emotions', 'Excessive Intensity', 'Emotional Deception', 'Missing Commitment', 'Missing Seriousness', 'Missing Openness', 'Missing Intelligibility', 'Unclear Meaning', 'Missing Relevance', 'Confusing Reasoning', 'Other Reasons', 'Detrimental Orthography', 'Reason Unclassified', 'fold0.0', 'split', 'global_row_id', 'p_appropriate', 'p_inappropriate', 'predicted_label']


,post_id,source_dataset,issue,post_text,Inappropriateness,Toxic Emotions,Excessive Intensity,Emotional Deception,Missing Commitment,Missing Seriousness,...,Confusing Reasoning,Other Reasons,Detrimental Orthography,Reason Unclassified,fold0.0,split,global_row_id,p_appropriate,p_inappropriate,predicted_label
0,8,0,Tv is better than books:,I thick that book are better than TV is it is ...,1,0,0,0,0,0,...,0,1,0,1,TEST,test,1754,0.002525,0.997475,LABEL_1
1,9,0,Tv is better than books:,Books enlighten the soul. Books don't destroy ...,1,1,1,1,1,0,...,1,0,0,0,TEST,test,1755,0.001745,0.998255,LABEL_1
2,12,0,India has the potential to lead the world:,india is is good adopter but a bad developer a...,1,0,0,0,1,0,...,1,0,0,0,TEST,test,1756,0.004373,0.995627,LABEL_1
3,18,0,William farquhar ought to be honoured as the r...,Raffles had written his discovery wrongly as 2...,1,0,0,0,0,0,...,1,0,0,0,TEST,test,1758,0.037113,0.962887,LABEL_1
4,22,0,Evolution vs creation:,"I am a nurse and the more I studied biology, t...",1,0,0,0,0,0,...,1,0,0,0,TEST,test,1759,0.007736,0.992264,LABEL_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1054,2072,0,Ban plastic water bottles:,this is a load of bull excrament. These argume...,1,1,1,1,1,1,...,0,0,0,0,VALID,validation,1742,0.001554,0.998446,LABEL_1
1055,2081,0,Gay marriage right or wrong:,Is incestual marriage right or wrong? Does gay...,1,0,0,0,1,1,...,0,0,0,0,VALID,validation,1744,0.002798,0.997202,LABEL_1
1056,2083,0,Gay marriage right or wrong:,I am a christian. The bible says plain out tha...,1,0,0,0,1,0,...,0,0,0,0,VALID,validation,1745,0.001641,0.998359,LABEL_1
1057,2112,1,Should freedom of speech be of unlimited propo...,"Yes, it should be of unlimited proportions. Fr...",1,1,1,0,1,0,...,0,0,0,0,VALID,validation,1746,0.003510,0.996490,LABEL_1


In [2]:
required_cols = [
    "global_row_id",
    "Inappropriateness",
    "p_inappropriate",
    "post_text"
]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

# Eine Zeile pro Argument
base = df.drop_duplicates(subset=["global_row_id"]).copy()

# True positive aus Sicht deines Document-Level-Classifiers
tp = base[
    (base["Inappropriateness"] == 1) &
    (base["p_inappropriate"] >= 0.5)
].copy()

print("True positives:", len(tp))

sample_100 = (
    tp.sample(n=min(N_SAMPLE, len(tp)), random_state=RANDOM_SEED)
      .reset_index(drop=True)
)

sample_path = Path("results/llm_reference")
sample_path.mkdir(parents=True, exist_ok=True)

sample_100.to_csv(sample_path / "sample_100_true_positives.csv", index=False)

sample_100[["global_row_id", "post_id", "split", "issue", "post_text", "p_inappropriate"]].head()

True positives: 1059


,global_row_id,post_id,split,issue,post_text,p_inappropriate
0,730,1052,train,Why isn't prostitution legal?:,"I certainly don't fear ""the competition""...Bes...",0.998349
1,235,341,train,Evolution vs creation:,"If you can, then please do so. Illuminate us w...",0.998406
2,1921,791,test,India has the potential to lead the world:,this does not even make sense no one country s...,0.997772
3,958,1381,train,"Cmv: the ""war on terror™"" the biggest scam in ...","I'm 30, and it seems like for nearly have my l...",0.997734
4,1395,2004,train,Cmv: the term redskin is not equivalent to the...,Throughout the debate with the Washington Reds...,0.997757


# API Setup

In [3]:
import os
from dotenv import load_dotenv
import json
import time
import re
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from google import genai
from google.genai import types
from google.api_core.exceptions import ResourceExhausted

load_dotenv(".env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL_NAME = "gemini-3-flash-preview"
BATCH_SIZE = 5
SECONDS_BETWEEN_SUCCESSFUL_BATCHES = 30
MAX_RETRIES_PER_BATCH = 20

OUTPUT_DIR = Path("results/llm_reference")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "llm_span_annotations_100_tp_gemini_batched.csv"

In [4]:
BATCH_SCHEMA = {
    "type": "object",
    "properties": {
        "annotations": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "argument_id": {"type": "string"},
                    "span_text": {"type": "string"},
                    "char_start": {"type": "integer"},
                    "char_end": {"type": "integer"},
                    "confidence": {"type": "number"}
                },
                "required": [
                    "argument_id",
                    "span_text",
                    "char_start",
                    "char_end",
                    "confidence"
                ]
            }
        }
    },
    "required": ["annotations"]
}

In [5]:
SYSTEM_PROMPT_BATCH = """
You are annotating argumentative texts for inappropriateness.

Each input argument is already labeled as inappropriate.
For each argument, identify exactly one minimal text span that best explains why the argument may be considered inappropriate.

A relevant span may indicate, for example:
- toxic or insulting language,
- excessive emotional intensity,
- missing seriousness,
- missing openness to discussion,
- unclear meaning,
- missing relevance to the issue,
- or another flaw that makes the argumentative language inappropriate.

Annotation rules:
1. Return exactly one span per argument.
2. The span must occur verbatim in the original argument text.
3. Do not paraphrase.
4. Prefer a minimal span over a whole sentence.
5. Do not select neutral context unless it is necessary.
6. Character offsets must refer to the original argument text.
7. Output valid JSON only.
8. Return one annotation object for every input argument_id.
"""

In [6]:
def extract_retry_delay_seconds(error_message, default=120):
    """
    Extracts retry delay from Gemini error messages.
    Falls back to default seconds.
    """
    text = str(error_message)

    # Example: "retryDelay': '15s'"
    match = re.search(r"retryDelay['\"]?: ['\"]?(\d+)s", text)
    if match:
        return int(match.group(1)) + 10

    # Example: "Please retry in 15.858879528s"
    match = re.search(r"retry in ([0-9.]+)s", text)
    if match:
        return int(float(match.group(1))) + 10

    return default


def annotate_one_argument(row, text_col=TEXT_COL):
    argument_text = str(row[text_col])

    user_input = {
        "argument_id": str(row["global_row_id"]),
        "issue": str(row.get("issue", "")),
        "argument": argument_text
    }

    prompt = SYSTEM_PROMPT_BATCH + "\n\nInput:\n" + json.dumps(user_input, ensure_ascii=False)

    last_error = None

    for attempt in range(1, MAX_RETRIES_PER_BATCH + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                    response_schema={
                        "type": "object",
                        "properties": {
                            "span_text": {"type": "string"},
                            "char_start": {"type": "integer"},
                            "char_end": {"type": "integer"},
                            "confidence": {"type": "number"}
                        },
                        "required": [
                            "span_text",
                            "char_start",
                            "char_end",
                            "confidence"
                        ]
                    }
                )
            )

            return json.loads(response.text)

        except ResourceExhausted as e:
            last_error = e
            wait_seconds = extract_retry_delay_seconds(e, default=20)

            print(
                f"429 RESOURCE_EXHAUSTED at argument {row['global_row_id']} "
                f"(attempt {attempt}/{MAX_RETRIES_PER_BATCH}). Waiting {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

        except Exception as e:
            last_error = e

            # Bei anderen temporären Fehlern kurzer Backoff
            wait_seconds = min(60, 5 * attempt)

            print(
                f"Error at argument {row['global_row_id']} "
                f"(attempt {attempt}/{MAX_RETRIES}): {e}. "
                f"Waiting {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        f"Failed after {MAX_RETRIES_PER_BATCH} retries for argument {row['global_row_id']}: {last_error}"
    )


def validate_and_fix_span(argument_text, span_text, char_start, char_end):
    argument_text = str(argument_text)
    span_text = str(span_text)

    result = {
        "valid": False,
        "fixed": False,
        "char_start": None,
        "char_end": None,
        "error": None
    }

    if not span_text.strip():
        result["error"] = "empty_span"
        return result

    # Case 1: Offsets stimmen exakt
    if (
        isinstance(char_start, int)
        and isinstance(char_end, int)
        and 0 <= char_start < char_end <= len(argument_text)
        and argument_text[char_start:char_end] == span_text
    ):
        result.update({
            "valid": True,
            "fixed": False,
            "char_start": char_start,
            "char_end": char_end
        })
        return result

    # Case 2: Spantext im Argument suchen
    first_idx = argument_text.find(span_text)

    if first_idx != -1:
        result.update({
            "valid": True,
            "fixed": True,
            "char_start": first_idx,
            "char_end": first_idx + len(span_text)
        })
        return result

    # Case 3: stripped span suchen
    stripped = span_text.strip()
    first_idx = argument_text.find(stripped)

    if first_idx != -1:
        result.update({
            "valid": True,
            "fixed": True,
            "char_start": first_idx,
            "char_end": first_idx + len(stripped)
        })
        return result

    result["error"] = "span_not_found"
    return result

def build_batch_payload(batch_df, text_col):
    arguments = []

    for _, row in batch_df.iterrows():
        arguments.append({
            "argument_id": str(row["global_row_id"]),
            "issue": str(row.get("issue", "")),
            "argument": str(row[text_col])
        })

    return {
        "arguments": arguments
    }

def annotate_batch_gemini_with_retry(batch_df, text_col):
    payload = build_batch_payload(batch_df, text_col)

    prompt = (
        SYSTEM_PROMPT_BATCH
        + "\n\nInput:\n"
        + json.dumps(payload, ensure_ascii=False)
    )

    last_error = None

    for attempt in range(1, MAX_RETRIES_PER_BATCH + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                    response_schema=BATCH_SCHEMA
                )
            )

            parsed = json.loads(response.text)

            if "annotations" not in parsed:
                raise ValueError(f"No 'annotations' key in response: {parsed}")

            annotations = parsed["annotations"]

            expected_ids = set(batch_df["global_row_id"].astype(str))
            returned_ids = set(str(a.get("argument_id")) for a in annotations)

            missing_ids = expected_ids - returned_ids

            if missing_ids:
                raise ValueError(f"Missing annotations for ids: {missing_ids}")

            return annotations

        except ResourceExhausted as e:
            last_error = e
            retry_delay = extract_retry_delay_seconds(e, default=180)

            # Exponentieller Backoff zusätzlich
            wait_seconds = max(retry_delay, min(900, 60 * attempt))

            print(
                f"429 RESOURCE_EXHAUSTED for batch "
                f"{batch_df['global_row_id'].tolist()} "
                f"(attempt {attempt}/{MAX_RETRIES_PER_BATCH}). "
                f"Waiting {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

        except Exception as e:
            last_error = e
            wait_seconds = min(300, 20 * attempt)

            print(
                f"Error for batch {batch_df['global_row_id'].tolist()} "
                f"(attempt {attempt}/{MAX_RETRIES_PER_BATCH}): {e}. "
                f"Waiting {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        f"Batch failed after {MAX_RETRIES_PER_BATCH} retries. Last error: {last_error}"
    )

def process_batch_annotations(batch_df, annotations, text_col):
    ann_by_id = {
        str(a["argument_id"]): a
        for a in annotations
    }

    rows = []

    for _, row in batch_df.iterrows():
        argument_id = str(row["global_row_id"])
        argument_text = str(row[text_col])

        ann = ann_by_id.get(argument_id)

        if ann is None:
            rows.append({
                "global_row_id": row["global_row_id"],
                "post_id": row.get("post_id", None),
                "split": row.get("split", None),
                "issue": row.get("issue", None),
                "argument": argument_text,
                "Inappropriateness": row["Inappropriateness"],
                "p_inappropriate": row["p_inappropriate"],

                "llm_span_text": None,
                "llm_char_start_original": None,
                "llm_char_end_original": None,
                "llm_char_start": None,
                "llm_char_end": None,
                "llm_confidence": None,

                "llm_valid": False,
                "llm_fixed_offsets": False,
                "llm_error": "missing_annotation",

                "model_name": MODEL_NAME
            })
            continue

        span_text = ann.get("span_text")
        char_start = ann.get("char_start")
        char_end = ann.get("char_end")
        confidence = ann.get("confidence")

        validation = validate_and_fix_span(
            argument_text=argument_text,
            span_text=span_text,
            char_start=char_start,
            char_end=char_end
        )

        rows.append({
            "global_row_id": row["global_row_id"],
            "post_id": row.get("post_id", None),
            "split": row.get("split", None),
            "issue": row.get("issue", None),
            "argument": argument_text,
            "Inappropriateness": row["Inappropriateness"],
            "p_inappropriate": row["p_inappropriate"],

            "llm_span_text": span_text,
            "llm_char_start_original": char_start,
            "llm_char_end_original": char_end,
            "llm_char_start": validation["char_start"],
            "llm_char_end": validation["char_end"],
            "llm_confidence": confidence,

            "llm_valid": validation["valid"],
            "llm_fixed_offsets": validation["fixed"],
            "llm_error": validation["error"],

            "model_name": MODEL_NAME
        })

    return rows

def load_existing_results(output_path):
    if output_path.exists():
        existing = pd.read_csv(output_path)
        print(f"Loaded existing checkpoint: {len(existing)} rows")
        return existing
    else:
        return pd.DataFrame()
    
def save_results(rows, output_path):
    result_df = pd.DataFrame(rows)

    # Falls durch Wiederholungen IDs mehrfach vorhanden sind:
    # valide Ergebnisse bevorzugen.
    if len(result_df) > 0:
        result_df["_valid_sort"] = result_df["llm_valid"].astype(bool).astype(int)
        result_df = (
            result_df
            .sort_values(["global_row_id", "_valid_sort"])
            .drop_duplicates(subset=["global_row_id"], keep="last")
            .drop(columns=["_valid_sort"])
            .sort_values("global_row_id")
        )

    result_df.to_csv(output_path, index=False)
    return result_df

In [7]:
def run_until_all_valid(sample_100, text_col, output_path):
    all_rows = []

    existing = load_existing_results(output_path)

    if len(existing) > 0:
        all_rows = existing.to_dict("records")

    target_ids = set(sample_100["global_row_id"].astype(str))

    round_idx = 1

    while True:
        current = pd.DataFrame(all_rows)

        if len(current) == 0:
            valid_ids = set()
        else:
            current["global_row_id_str"] = current["global_row_id"].astype(str)
            valid_ids = set(
                current[
                    current["llm_valid"].astype(bool)
                ]["global_row_id_str"]
            )

        missing_ids = target_ids - valid_ids

        print("=" * 80)
        print(f"Round {round_idx}")
        print(f"Valid: {len(valid_ids)}/{len(target_ids)}")
        print(f"Missing or invalid: {len(missing_ids)}")
        print("=" * 80)

        if len(missing_ids) == 0:
            print("All annotations are valid.")
            final_df = save_results(all_rows, output_path)
            return final_df

        todo = sample_100[
            sample_100["global_row_id"].astype(str).isin(missing_ids)
        ].copy()

        # Shuffle in later rounds can help if the model gets stuck on certain batches
        todo = todo.sample(frac=1, random_state=42 + round_idx).reset_index(drop=True)

        for start in tqdm(range(0, len(todo), BATCH_SIZE)):
            batch_df = todo.iloc[start:start + BATCH_SIZE].copy()

            batch_ids = batch_df["global_row_id"].tolist()
            print(f"Annotating batch ids: {batch_ids}")

            try:
                annotations = annotate_batch_gemini_with_retry(batch_df, text_col)
                batch_rows = process_batch_annotations(batch_df, annotations, text_col)

                all_rows.extend(batch_rows)
                current_saved = save_results(all_rows, output_path)

                n_valid = current_saved["llm_valid"].astype(bool).sum()
                print(f"Checkpoint saved. Valid now: {n_valid}/{len(target_ids)}")

            except Exception as e:
                print(f"Batch failed finally: {batch_ids}")
                print(e)

                # Fehlgeschlagene Batches nicht als final invalid speichern.
                # Sie bleiben dadurch in der nächsten Runde missing.
                current_saved = save_results(all_rows, output_path)

            print(
                f"Waiting {SECONDS_BETWEEN_SUCCESSFUL_BATCHES}s "
                f"before next batch..."
            )
            time.sleep(SECONDS_BETWEEN_SUCCESSFUL_BATCHES)

        round_idx += 1

In [8]:
llm_ref = run_until_all_valid(
    sample_100=sample_100,
    text_col=TEXT_COL,
    output_path=OUTPUT_PATH
)

Loaded existing checkpoint: 50 rows
Round 1
Valid: 50/100
Missing or invalid: 50


  0%|          | 0/10 [00:00<?, ?it/s]

Annotating batch ids: [469, 1107, 344, 633, 1738]
Checkpoint saved. Valid now: 55/100
Waiting 30s before next batch...
Annotating batch ids: [314, 291, 1378, 712, 17]
Checkpoint saved. Valid now: 60/100
Waiting 30s before next batch...
Annotating batch ids: [1191, 965, 174, 157, 2176]
Checkpoint saved. Valid now: 65/100
Waiting 30s before next batch...
Annotating batch ids: [1822, 135, 773, 1218, 1955]
Checkpoint saved. Valid now: 70/100
Waiting 30s before next batch...
Annotating batch ids: [566, 278, 758, 623, 539]
Checkpoint saved. Valid now: 75/100
Waiting 30s before next batch...
Annotating batch ids: [1405, 1406, 88, 1355, 684]
Checkpoint saved. Valid now: 80/100
Waiting 30s before next batch...
Annotating batch ids: [1279, 510, 368, 1670, 1155]
Checkpoint saved. Valid now: 85/100
Waiting 30s before next batch...
Annotating batch ids: [567, 915, 1072, 1217, 1493]
Checkpoint saved. Valid now: 90/100
Waiting 30s before next batch...
Annotating batch ids: [182, 1744, 1437, 2168, 458

In [9]:
llm_ref = pd.read_csv(OUTPUT_PATH)

print("Total rows:", len(llm_ref))
print("Valid counts:")
print(llm_ref["llm_valid"].value_counts(dropna=False))

display(llm_ref[[
    "global_row_id",
    "argument",
    "llm_span_text",
    "llm_char_start",
    "llm_char_end",
    "llm_confidence",
    "llm_valid",
    "llm_fixed_offsets",
    "llm_error"
]].head(20))

Total rows: 100
Valid counts:
llm_valid
True    100
Name: count, dtype: int64


,global_row_id,argument,llm_span_text,llm_char_start,llm_char_end,llm_confidence,llm_valid,llm_fixed_offsets,llm_error
0,8,"stupid india they actually really suck. so, BO...",stupid india they actually really suck,0,38,0.95,True,False,NaN
1,17,"Believing ""Evolution"" as in Darwinism and the ...",is like believing the puzzle can be solved by ...,52,163,0.80,True,False,NaN
2,49,There are tons of ways to not get pregnant. Th...,It's the woman's fault,206,228,0.90,True,True,NaN
3,53,If you do not like gay marriage and thing it i...,keep them to yourself,286,307,0.90,True,False,NaN
4,88,"These days, we grant more rights to convict's ...",we would all be better off if you were dead,474,517,0.90,True,False,NaN
5,95,I hear all too often that science is 'playing ...,"If god did not intend us to do so, he is nears...",576,671,1.00,True,True,NaN
6,96,Unless every single gun that is issued legally...,It's bull sh*t.,403,418,1.00,True,True,NaN
7,135,I note that whenever someone starts in on the ...,"""Feminist"" bullsh*t",271,290,0.95,True,False,NaN
8,145,If being homosexual was natural then\r\n1. The...,piss ant liberal,455,471,0.95,True,True,NaN
9,157,for everyone who is talking about RAPE in this...,you need to see a Physiatrist!!!!,300,333,0.90,True,False,NaN
